# Traffic Calibration -- evaluation notebook

Splits `OBSERVED_TRIPS` into 80% train / 20% hold-out by trip-start time, computes baseline and calibrated MAPE, and asserts a >=10% relative MAPE reduction on the hold-out (issue #64 acceptance criterion).

Prereq: `traffic-calibration/references/dynamic-tables.sql` has been deployed and the `SPEED_FACTORS` dynamic table has refreshed at least once.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.sql("""
ALTER SESSION SET query_tag = '{\"origin\":\"sf_sit-is-fleet\",\"name\":\"oss-traffic-calibration\",\"version\":{\"major\":1,\"minor\":0},\"attributes\":{\"is_quickstart\":1,\"source\":\"notebook\"}}'
""").collect()

In [ ]:
# 80 / 20 split by TRIP_START_TIME. Time-based split (not random) so the
# eval mimics deploying calibration today and using it on tomorrow's trips.
trips = session.sql("""
WITH ranked AS (
    SELECT *,
           PERCENT_RANK() OVER (ORDER BY TRIP_START_TIME) AS pct_rank
      FROM FLEET_INTELLIGENCE.TRAFFIC.OBSERVED_TRIPS
     WHERE DURATION_RATIO IS NOT NULL
       AND DURATION_RATIO BETWEEN 0.3 AND 3.0
)
SELECT *, CASE WHEN pct_rank < 0.8 THEN 'train' ELSE 'holdout' END AS split
  FROM ranked
""").to_pandas()

holdout = trips[trips['SPLIT'] == 'holdout'].copy()
print(f'train rows: {(trips["SPLIT"] == "train").sum():,}')
print(f'holdout rows: {len(holdout):,}')


In [ ]:
# Compute calibrated durations for the holdout using the deployed UDF.
# (We could also join SPEED_FACTORS directly but going through the UDF
#  exercises the exact code path the demos will use.)
session.sql("""
CREATE OR REPLACE TEMPORARY TABLE _HOLDOUT_EVAL AS
SELECT
    t.TRIP_ID,
    t.OBSERVED_DURATION_SEC,
    t.ORS_PREDICTED_DURATION_SEC                              AS BASELINE_DURATION_SEC,
    FLEET_INTELLIGENCE.TRAFFIC.CALIBRATED_DURATION(
        t.PROFILE, t.ORS_PREDICTED_DURATION_SEC, t.HOUR_OF_DAY, t.ROAD_CLASS, t.REGION
    )                                                         AS CALIBRATED_DURATION_SEC
FROM FLEET_INTELLIGENCE.TRAFFIC.OBSERVED_TRIPS t
WHERE t.TRIP_START_TIME >= (
    SELECT APPROX_PERCENTILE(TRIP_START_TIME, 0.8) FROM FLEET_INTELLIGENCE.TRAFFIC.OBSERVED_TRIPS
)
""").collect()

results = session.sql("""
SELECT
    AVG(ABS(BASELINE_DURATION_SEC   - OBSERVED_DURATION_SEC) / OBSERVED_DURATION_SEC) AS BASELINE_MAPE,
    AVG(ABS(CALIBRATED_DURATION_SEC - OBSERVED_DURATION_SEC) / OBSERVED_DURATION_SEC) AS CALIBRATED_MAPE
  FROM _HOLDOUT_EVAL
""").to_pandas().iloc[0]

baseline   = float(results['BASELINE_MAPE'])
calibrated = float(results['CALIBRATED_MAPE'])
reduction  = 100 * (baseline - calibrated) / baseline

print(f'Baseline   MAPE: {baseline:.3f}')
print(f'Calibrated MAPE: {calibrated:.3f}')
print(f'Reduction:       {reduction:.1f}%')

assert reduction >= 10, (
    f'MAPE reduction {reduction:.1f}% below target 10%. '
    f'Try adding DOW to the bucket key in references/dynamic-tables.sql.'
)
print('PASS: >=10% MAPE reduction on hold-out')

In [ ]:
# Bucket coverage report -- helps decide whether to refine the key.
coverage = session.sql("""
SELECT
    PROFILE,
    HOUR_OF_DAY,
    ROAD_CLASS,
    SPEED_FACTOR,
    TRIP_COUNT
  FROM FLEET_INTELLIGENCE.TRAFFIC.SPEED_FACTORS
 ORDER BY TRIP_COUNT DESC
 LIMIT 50
""").to_pandas()
coverage